Comparing the diversity of narrative reports for reports with the same event tags to assess inter-rater reliability

In [2]:
import pandas as pd
import numpy as np

from datasets import load_dataset

from huggingface_hub import hf_hub_download

import utils 

/home/maciej/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
events_df = pd.read_csv("../data-processing/events.csv")
events_df.rename(columns={"Unnamed: 0" : "acn"}, inplace=True)
#events_df.set_index("acn", inplace=True)

combined_events_df = events_df[events_df.columns[:1]]

# Remove NaN instances from set
combined_events_df["temp"] = events_df[events_df.columns[2:]].agg(set, axis=1)
for index, row in combined_events_df.iterrows():
    no_nan_list = []
    for event in row["temp"]: 
        if isinstance(event, str): no_nan_list.append(event)
    combined_events_df.at[index, "temp"] = no_nan_list

combined_events_df

# Converted to a string because strings can be hashed, which is useful for finding duplicates
combined_events_df["events"] = [", ".join(r["temp"]) for i, r in combined_events_df.iterrows()]
combined_events_df.drop(columns="temp", inplace=True)
combined_events_df

/tmp/ipykernel_2743/3739016954.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  combined_events_df["temp"] = events_df[events_df.columns[2:]].agg(set, axis=1)
/tmp/ipykernel_2743/3739016954.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  combined_events_df["events"] = [", ".join(r["temp"]) for i, r in combined_events_df.iterrows()]
/tmp/ipykernel_2743/3739016954.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: ht

,acn,events
0,2260174,anomaly_deviation_/_discrepancy_-_procedural_p...
1,2260249,anomaly_inflight_event_/_encounter_cftt_/_cfit...
2,2260370,"result_flight_crew_diverted, result_flight_cre..."
3,2261277,anomaly_deviation_/_discrepancy_-_procedural_p...
4,2261317,anomaly_deviation_/_discrepancy_-_procedural_p...
...,...,...
24529,2314733,"anomaly_aircraft_equipment_problem_critical, a..."
24530,2314971,"passenger_involvement_n, detector_person_fligh..."
24531,2314972,anomaly_deviation_/_discrepancy_-_procedural_f...
24532,2314978,anomaly_deviation_/_discrepancy_-_procedural_p...


In [4]:
narratives_df = pd.read_csv("../data-processing/narratives.csv")
narratives_df.rename(columns={"Unnamed: 0" : "acn"}, inplace=True)
narratives_df.set_index("acn", inplace=True)

narratives_df.head()

,Assessments_Primary_Problem,Report_1_Narrative,Report_2_Narrative
acn,,,
2260174,ambiguous,Aircraft vectored in within 1NM to final appro...,NaN
2260249,ambiguous,While on short final we received a glideslope ...,NaN
2260370,aircraft,Flying at cruise; FL350; the FO was the PF and...,At cruise; FL350; during level-off; the Captai...
2261277,humanfactors,On Day 0 around XA:30; I forgot to get LAANC a...,NaN
2261317,procedure,Divert into ZZZ from ZZZ1. FO flying. Vectors ...,Extremely strong winds blew us off the LOC whi...


In [5]:
# Group by the same events string, and then turn those ACNs into lists
duplicates_series = combined_events_df.groupby("events")["acn"].apply(list)
duplicates_series

for index, acns in duplicates_series.items():
    if len(acns) <= 1:          # Not a duplicate
        duplicates_series.drop(index, inplace=True)

duplicates_series

events
anomaly_aircraft_equipment_problem_critical                                                                                                                                                                                                                                                                                                                                                       [2032791, 2260241, 2261197, 2264946, 2270182, ...
anomaly_aircraft_equipment_problem_critical, anomaly_deviation_/_discrepancy_-_procedural_clearance                                                                                                                                                                                                                                                                                                   [2285608, 2296521, 2298943, 2304776, 2314733]
anomaly_aircraft_equipment_problem_critical, detector_automation_aircraft_other_automation, result_flight_crew_flc_compli

### Analysis

In [6]:
def printDuplicateNarratives(index : int):
    print("Event Tags:")
    tags = duplicates_series.index[index].split(", ")
    
    for t in tags:
        print('\t', t)

    print()

    for i in range(len(duplicates_series.iloc[index])):
        print(f"===== Narrative {i} =====")
        print(narratives_df.at[duplicates_series.iloc[index][i], "Report_1_Narrative"])
        print()

In [ ]:
# Note: if you're using VS Code like I am, you can turn on word wrapping in settings under notebook.output.wordWrap

printDuplicateNarratives(10)

Event Tags:
	 anomaly_aircraft_equipment_problem_critical
	 result_flight_crew_returned_to_departure_airport
	 result_flight_crew_landed_in_emergency_condition
	 result_flight_crew_overcame_equipment_problem
	 detector_automation_aircraft_other_automation
	 when_detected_in_flight

===== Narrative 0 =====
Gear disagree EICAS MSG after takeoff. [Priority handling requested]/returning to ZZZ. Initially overweight landing by 15000lb. During configuration for landing no green indication.  For the R main gear after extension. Manual extension attempted and was unsuccessful. It is in question whether the aircraft gear is down and locked.R side brace EICAS MSG illuminated; gear door EICAS MSG illuminated; & gear disagree MSG extinguishedA conference via SATCOM was initiated with Maintenance Control and Chief Pilot. Attempts were made by Chief Pilot to have Boeing tech join the patch; however this was not successful. G force maneuvers were recommended by Maintenance Control yet unsuccessful yi